In [ ]:
# IL-only GPU AdamW (PTX, khong C++) - 1/4: GPU + mono (KHONG build C/C++)
import os, subprocess, glob, re
r=subprocess.run('wget -q https://dot.net/v1/dotnet-install.sh -O /tmp/dotnet-install.sh && bash /tmp/dotnet-install.sh --channel 10.0 --install-dir /kaggle/working/dotnet', shell=True, capture_output=True, text=True, timeout=1200)
print('dotnet install rc=', r.returncode, (r.stderr or '')[-400:])
v=subprocess.run('/kaggle/working/dotnet/dotnet --version', shell=True, capture_output=True, text=True)
print((v.stdout or '').split('\n')[0])
print('KHONG import torch - duong IL-only doc lap')

In [ ]:
# 2/4: clone backend + symlink CUDA libs (Kaggle GPU)
import os, subprocess, glob
d='/kaggle/working/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
r=subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
print('clone rc=', r.returncode)
print('CUDA backend HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD',shell=True).decode().strip())
libdir='/kaggle/working/tkvlib'; os.makedirs(libdir, exist_ok=True)
def pick(pats):
    for pat in pats:
        f=[p for p in glob.glob(pat) if os.path.exists(os.path.realpath(p))]
        if f: return sorted(f)[0]
    return None
libs={
 'cudart.dll': pick(['/usr/local/cuda*/lib64/libcudart.so','/usr/lib/x86_64-linux-gnu/libcudart.so*']),
 'cublas.dll': pick(['/usr/local/cuda*/lib64/libcublas.so','/usr/lib/x86_64-linux-gnu/libcublas.so*']),
 'cuda.dll': pick(['/usr/lib/x86_64-linux-gnu/libcuda.so*','/usr/lib64/libcuda.so*','/usr/local/nvidia/lib64/libcuda.so*','/usr/local/nvidia/lib/libcuda.so*','/usr/lib64-nvidia/libcuda.so*','/usr/local/cuda/compat/libcuda.so*'])
}
for nm,tgt in libs.items():
    print(nm,'->',tgt)
    if tgt:
        p=libdir+'/'+nm
        if os.path.lexists(p): os.remove(p)
        os.symlink(tgt,p)
os.environ['LD_LIBRARY_PATH']=libdir+':'+os.environ.get('LD_LIBRARY_PATH','')
import ctypes.util
print('find_library cuda =', ctypes.util.find_library('cuda'))

In [ ]:
# 3/4: chay AdamW PTX kernel via .NET 10
import subprocess, os
exe=d+'/Release/cuda_backend/examples/ilgpu_adamw.exe'
print('exe exists=',os.path.exists(exe))
rcfg=os.path.dirname(exe)+'/ilgpu_adamw.runtimeconfig.json'
open(rcfg,'w').write('{"runtimeOptions":{"tfm":"net10.0","framework":{"name":"Microsoft.NETCore.App","version":"10.0.0"}}}')
r=subprocess.run(['/kaggle/working/dotnet/dotnet',exe], env=dict(os.environ), capture_output=True, text=True, timeout=300)
print('dotnet rc=',r.returncode)
print(r.stdout[:1500])
print('STDERR:',(r.stderr or '')[:1500])

In [ ]:
# 4/4: ket luan don gian
import re
vals=[float(v) for v in re.findall(r'^ADAMW\s+\d+\s+([0-9.eE+-]+)$',r.stdout,re.M)]
print('ADAMW n=',len(vals),' first=',vals[:3] if vals else None)